# Theorem 4 — observed-private pathwise invariance

**Formal source:** [`../04_observed_private_invariance.md`](../04_observed_private_invariance.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
low = np.diag([1, 1, 0, 0, 0])
high_a = np.diag([1, 1, 1, 0, 0])
high_b = np.diag([1, 1, 0, 1, 0])
shared, private, missing, global_null = four_way([low, high_a, high_b], 1)
state = np.array([1.0, -1.0, 7.0, -0.5, 9.0])
initial_private = private @ state
initial_null = global_null @ state
for _ in range(100):
    state += 0.01 * (shared @ (-0.2 * (shared @ state)) + missing @ (-0.1 * (missing @ state) + 0.05 * (shared @ state)))
    state += missing @ np.array([0, 0, 0, 0.001, 0])
np.testing.assert_allclose(private @ state, initial_private, atol=1e-12)
np.testing.assert_allclose(global_null @ state, initial_null, atol=1e-12)
leaking = state + np.array([0, 0, 0.2, 0, 0])
assert np.linalg.norm(private @ leaking - initial_private) > 0.1
print({"private_drift": float(np.linalg.norm(private @ state - initial_private)), "negative_control": float(np.linalg.norm(private @ leaking - initial_private))})

In [ ]:
print('THEORY_DEMO_PASS::04_observed_private_invariance')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')